# Class 15: Thirty-Five Worlds
*Elements of Data Science — Honors*

**First thing:** save this notebook under a new name that includes your team's name.

Parts 1 is on paper and already done. This notebook is Parts 2 and 3.

In [ ]:
from datascience import *
import numpy as np
# import for plotting
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
# Fix for datascience plots
import collections as collections
import collections.abc as abc
collections.Iterable = abc.Iterable

---
## Part 2. Why Bother Simulating?

The seven patients you just worked with by hand.

In [ ]:
experiment = Table().with_columns(
    'Patients', make_array('Control', 'Control', 'Control',
                           'Treated', 'Treated', 'Treated', 'Treated'),
    'Recovery Time (days)', make_array(22, 33, 40, 19, 22, 25, 26)
)

experiment.group('Patients', np.mean)

### The observed difference

Compute it from the data rather than typing in a number off the screen. The two are not the
same: the printed mean is rounded, and later we ask which permutations are *at least as
extreme as the observed one*. A threshold that is a hair too big quietly drops the observed
value out of its own tail and shrinks the p-value.

In [ ]:
control = experiment.where('Patients', 'Control').column('Recovery Time (days)')
treated = experiment.where('Patients', 'Treated').column('Recovery Time (days)')

observed = np.mean(control) - np.mean(treated)
observed

**2.1** All 35 splits, in code. `combinations` hands back every way to choose 3 of the
7 recovery times; the other 4 are the treated group.

In [ ]:
from itertools import combinations

times = experiment.column('Recovery Time (days)')
all_diffs = make_array()

for control_positions in combinations(np.arange(7), 3):
    in_control = np.array([i in control_positions for i in np.arange(7)])
    d = np.mean(times[in_control]) - np.mean(times[~in_control])
    all_diffs = np.append(all_diffs, d)

print('number of splits:', len(all_diffs))
print('mean difference: ', np.round(np.mean(all_diffs), 10))

Now the two tails. `>=` alone asks whether the treatment helped; `np.abs` asks
whether it did anything at all. Which one you are entitled to use was decided before you saw
the data — on the worksheet, in 1.5.

In [ ]:
upper = all_diffs >= observed
both  = np.abs(all_diffs) >= np.abs(observed)

print('treated did this much better or more:', np.count_nonzero(upper))
print('either group did, by this much:      ', np.count_nonzero(both))
print()
print('one-sided p:', np.count_nonzero(upper) / 35)
print('two-sided p:', np.count_nonzero(both) / 35)

Compare both with the board. They should agree exactly — same 35 worlds, same counts.

Notice that the two-sided p-value is not twice the one-sided one. Doubling works when the
null distribution is symmetric, and this one is not: the groups are different sizes, 3 and 4,
so the differences run further in one direction than the other. Check it:

In [ ]:
print('most negative difference:', np.round(min(all_diffs), 2))
print('largest difference:      ', np.round(max(all_diffs), 2))



**2.2** Now the version you would have to use if enumeration were impossible: shuffle the
labels at random, many times, and see how often chance beats what you observed.

In [ ]:
shuffled_diffs = make_array()

for i in np.arange(5000):
    shuffled = np.random.permutation(times)
    d = np.mean(shuffled[:3]) - np.mean(shuffled[3:])
    shuffled_diffs = np.append(shuffled_diffs, d)

np.count_nonzero(np.abs(shuffled_diffs) >= np.abs(observed)) / 5000

Run that cell three times and record all three answers. Then compare them with the
exact value from 2.1 — the simulation is an estimate of a number you already know precisely,
which is the only situation in which you can see how good the estimate is.

**2.3** So why ever simulate? Because seven patients is a toy.

Seven patients had 35 possible splits. A realistic trial — 70 patients, 30 control and 40
treated — has about **5.5 × 10¹⁹** of them. Take that number as given; counting it is a
problem for another course.

Suppose a computer could check a million splits every second. Work out how long the
exhaustive version would take, in years.

In [ ]:
splits = 5.5e19
per_second = 1e6

seconds = splits / per_second
years = ...

years

Record that number on the worksheet, and work out how long the exhaustive version
would take at a million splits per second.

---
## Part 3. What a p-value Does When Nothing Is Happening

A p-value is not a property of a treatment. It is a number computed from data, and data is
random, so the p-value is random too. The only way to see the shape of that randomness is to
run the whole experiment over and over.

The function below runs one complete trial from scratch: it invents 12 control patients and
12 treated patients, computes the difference, and permutation-tests it 200 times. It returns
the p-value of that single trial.

`effect` is how many days the treatment truly saves. Set it to 0 and the treatment does
**nothing** — not "we cannot detect anything," but nothing, by construction.

In [ ]:
def run_one_experiment(effect):
    control = np.random.normal(30, 6, 12)
    treated = np.random.normal(30 - effect, 6, 12)
    observed = np.mean(control) - np.mean(treated)

    pool = np.append(control, treated)
    extreme = 0
    for i in np.arange(200):
        np.random.shuffle(pool)
        d = np.mean(pool[:12]) - np.mean(pool[12:])
        if np.abs(d) >= np.abs(observed):
            extreme = extreme + 1

    return extreme / 200

In [ ]:
# One trial where the treatment does nothing
run_one_experiment(0)

**3.1** Write your prediction on the worksheet before running the next cell. Five
hundred trials, in every one of which the treatment does nothing. This takes a few seconds.

In [ ]:
null_p_values = make_array()

for i in np.arange(500):
    null_p_values = np.append(null_p_values, run_one_experiment(0))

print('p-values collected:', len(null_p_values), '  (should be 500)')

In [ ]:
Table().with_column('p-value', null_p_values).hist(bins=np.arange(0, 1.01, 0.05))
plt.title('500 trials in which the treatment did nothing');

In [ ]:
# What fraction came out "statistically significant"?
np.mean(null_p_values < 0.05)

**3.2 and 3.3** Record the shape and that fraction. Then account for those significant
results, given that nothing was wrong with any of the 500 trials.

**3.4** Now a treatment that really works — five days faster, every time.

In [ ]:
effect_p_values = make_array()

for i in np.arange(500):
    ...

print('p-values collected:', len(effect_p_values), '  (should be 500)')

In [ ]:
Table().with_column('p-value', effect_p_values).hist(bins=np.arange(0, 1.01, 0.05))
plt.title('500 trials with a real 5-day effect');

In [ ]:
np.mean(effect_p_values < 0.05)

The effect was there in all 500 trials. Record how often the test found it, and answer
the last question on the worksheet.